# Friend A - B5 test-time adaptation (resumable)

Current task from `CONTEXT.md section 14`: evaluate adaptation on **unlabelled target-embryo images** with the honest `6bba -> 44b6` model. The old dense fine-tune is no longer Friend A's active queue.

The notebook evaluates three modes on the fixed 12-video 44b6 set:

1. unchanged `friend_a_exp02_loeo20` checkpoint;
2. per-video BatchNorm statistics recalibration;
3. per-video detection-head-only EMA self-training with mild photometric consistency.

Adaptation opens only `.zarr` images. The separate official scoring stage reads `.geff` labels only after every prediction for a variant exists. Every video starts from the same base weights. Nothing is submitted to the leaderboard.

## Stop and resume

- Click **Interrupt Kernel**. Head-EMA is checkpointed every 25 steps and when interruption is caught.
- Or create `C:\\Projects\\CellTracker\\outputs\\local_friend_a_exp04_tta_b5\\STOP_REQUESTED` from another PowerShell window.
- A repeated **Run All** skips completed videos and resumes an unfinished checkpoint.
- Before starting Jupyter, set `$env:BIOHUB_TTA_MAX_VIDEOS = "1"` for short slices or `$env:BIOHUB_TTA_RUN = "0"` for preflight only.


In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
candidates = [start, start / '.github-push-worktree']
for parent in start.parents:
    candidates.extend([parent, parent / '.github-push-worktree'])
REPO_ROOT = next((p for p in candidates if (p / 'TEAM.md').is_file() and (p / 'tools' / 'score_geffs.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Repository root not found. Start Jupyter from C:\\Projects\\CellTracker.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from experiments.local_friend_a_exp04_tta_b5.runner import (
    TTAConfig, clear_stop, compact_status, preflight, run_pipeline, stop_path,
)
print('Repository:', REPO_ROOT)


In [ ]:
# All settings can be overridden with BIOHUB_TTA_* before Jupyter starts.
cfg = TTAConfig.from_environment(REPO_ROOT)
cfg.serialisable()


In [ ]:
# Clears only a stale stop request; results and checkpoints remain intact.
clear_stop(cfg)
check = preflight(cfg, gpu_smoke=cfg.run_pipeline)
check


The next cell may run for several hours. It safely reuses completed graphs and checkpoints. Baseline is completed before AdaBN and head-EMA, so every adapted result uses the same baseline and an unsupervised node-count guard.


In [ ]:
state = run_pipeline(cfg)
compact_status(cfg)


In [ ]:
# The final table appears after all 12 graphs for a variant are complete.
status = compact_status(cfg)
print(status)
result_path = cfg.output_dir / 'result.md'
if result_path.is_file():
    print(result_path.read_text(encoding='utf-8'))
print('Stop marker for an external PowerShell:', stop_path(cfg))
